# Filtering Rows with Conditions in Pandas

This notebook is about **selecting rows that match a condition** — the everyday
job of a data analyst.

You will learn:

- How a condition becomes a `True`/`False` mask
- `df[mask]` vs the safer `df.loc[mask]`
- `df.loc[condition, columns]` — filter rows and pick columns in one step
- Combining conditions with `&` (and), `|` (or), `~` (not)
- Counting and averaging the rows that survive a filter

Dataset: `../data/titanic_dataset.csv`

In [1]:
import pandas as pd

df = pd.read_csv("../data/titanic_dataset.csv")
df.head(10)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     1309 non-null   int64  
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 122.8+ KB


In [3]:
# describe() needs the () — without them you only print the method object
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,1309.000000,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,655.000000,0.377387,2.294882,29.881138,0.498854,0.385027,33.295479
std,378.020061,0.484918,0.837836,14.413493,1.041658,0.865560,51.758668
min,1.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,328.000000,0.000000,2.000000,21.000000,0.000000,0.000000,7.895800
50%,655.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,982.000000,1.000000,3.000000,39.000000,1.000000,0.000000,31.275000
max,1309.000000,1.000000,3.000000,80.000000,8.000000,9.000000,512.329200


## 1. The idea behind every filter

A condition like `df["Age"] > 30` does **not** return rows.
It returns a **mask**: one `True`/`False` value per row.

Passing that mask back to the DataFrame keeps only the `True` rows.

```
condition  ->  True/False per row  ->  df.loc[mask]  ->  matching rows
```

## 2. The `.loc` syntax

```python
df.loc[row_condition, column_selection]
```

- left of the comma → **which rows** (a condition, a label, or `:` for all)
- right of the comma → **which columns** (a name, a list of names, or `:` for all)

Use `.loc` rather than `df[mask]` when you also want to pick columns, or when you
intend to **change** values — `.loc` writes straight into the original DataFrame.

In [4]:
df.loc[1, "Name"]

'Cumings, Mrs. John Bradley (Florence Briggs Thayer)'

In [5]:
df["Age"]>30

0       False
1        True
2       False
3        True
4        True
        ...  
1304    False
1305     True
1306     True
1307    False
1308    False
Name: Age, Length: 1309, dtype: bool

In [6]:
df[df["Age"]>30]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1295,1296,0,1,"Frauenthal, Mr. Isaac Gerald",male,43.0,1,0,17765,27.7208,D40,C
1298,1299,0,1,"Widener, Mr. George Dunton",male,50.0,1,1,113503,211.5000,C80,C
1302,1303,1,1,"Minahan, Mrs. William Edward (Lillian E Thorpe)",female,37.0,1,0,19928,90.0000,C78,Q
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C


In [7]:
df.loc[df["Age"] > 30, :]


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1295,1296,0,1,"Frauenthal, Mr. Isaac Gerald",male,43.0,1,0,17765,27.7208,D40,C
1298,1299,0,1,"Widener, Mr. George Dunton",male,50.0,1,1,113503,211.5000,C80,C
1302,1303,1,1,"Minahan, Mrs. William Edward (Lillian E Thorpe)",female,37.0,1,0,19928,90.0000,C78,Q
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C


In [8]:
df.loc[df["Fare"] > 50 , ["Fare", "Age", "Embarked"]]

,Fare,Age,Embarked
1,71.2833,38.0,C
3,53.1000,35.0,S
6,51.8625,54.0,S
27,263.0000,19.0,S
31,146.5208,NaN,C
...,...,...,...
1291,164.8667,30.0,S
1293,59.4000,22.0,C
1298,211.5000,50.0,C
1302,90.0000,37.0,Q


In [9]:
# Passengers who survived
df.loc[df["Survived"] == 1, ["Name","Age","Survived"]]


,Name,Age,Survived
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1
2,"Heikkinen, Miss. Laina",26.0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,1
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,1
...,...,...,...
1300,"Peacock, Miss. Treasteall",3.0,1
1301,"Naughton, Miss. Hannah",NaN,1
1302,"Minahan, Mrs. William Edward (Lillian E Thorpe)",37.0,1
1303,"Henriksson, Miss. Jenny Lovisa",28.0,1


In [10]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,0,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,0,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [11]:
df.loc[df["Survived"] == 0, ["Age","Survived", "Name", "Embarked"]]

,Age,Survived,Name,Embarked
0,22.0,0,"Braund, Mr. Owen Harris",S
4,35.0,0,"Allen, Mr. William Henry",S
5,NaN,0,"Moran, Mr. James",Q
6,54.0,0,"McCarthy, Mr. Timothy J",S
7,2.0,0,"Palsson, Master. Gosta Leonard",S
...,...,...,...,...
1298,50.0,0,"Widener, Mr. George Dunton",C
1304,NaN,0,"Spector, Mr. Woolf",S
1306,38.5,0,"Saether, Mr. Simon Sivertsen",S
1307,NaN,0,"Ware, Mr. Frederick",S


In [12]:
df["Pclass"].unique()

array([3, 1, 2])

In [13]:
#Passengers in 1st class
df.loc[df["Pclass"]==1,["Name","Age","Pclass","Sex"]]


,Name,Age,Pclass,Sex
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,female
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,female
6,"McCarthy, Mr. Timothy J",54.0,1,male
11,"Bonnell, Miss. Elizabeth",58.0,1,female
23,"Sloper, Mr. William Thompson",28.0,1,male
...,...,...,...,...
1294,"Carrau, Mr. Jose Pedro",17.0,1,male
1295,"Frauenthal, Mr. Isaac Gerald",43.0,1,male
1298,"Widener, Mr. George Dunton",50.0,1,male
1302,"Minahan, Mrs. William Edward (Lillian E Thorpe)",37.0,1,female


In [14]:
# Age > 30 AND Survived
df.loc[(df["Age"] > 30) & (df["Survived"] == 0), :]


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,NaN,S
18,19,0,3,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,31.0,1,0,345763,18.0000,NaN,S
20,21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.0000,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1290,1291,0,3,"Conlon, Mr. Thomas Henry",male,31.0,0,0,21332,7.7333,NaN,Q
1292,1293,0,2,"Gale, Mr. Harry",male,38.0,1,0,28664,21.0000,NaN,S
1295,1296,0,1,"Frauenthal, Mr. Isaac Gerald",male,43.0,1,0,17765,27.7208,D40,C
1298,1299,0,1,"Widener, Mr. George Dunton",male,50.0,1,1,113503,211.5000,C80,C


In [15]:
df["Sex"].unique()

array(['male', 'female'], dtype=object)

In [16]:
# Age < 18 OR Female

df.loc[(df["Age"] < 18) | (df["Sex"] == "female"), :]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1300,1301,1,3,"Peacock, Miss. Treasteall",female,3.0,1,1,SOTON/O.Q. 3101315,13.7750,NaN,S
1301,1302,1,3,"Naughton, Miss. Hannah",female,NaN,0,0,365237,7.7500,NaN,Q
1302,1303,1,1,"Minahan, Mrs. William Edward (Lillian E Thorpe)",female,37.0,1,0,19928,90.0000,C78,Q
1303,1304,1,3,"Henriksson, Miss. Jenny Lovisa",female,28.0,0,0,347086,7.7750,NaN,S


In [17]:
# name age pclass emabrked
# male pclass 3

df.loc[(df["Sex"] == "male") & (df["Pclass"] == 3), ["Name", "Age", "Pclass", "Embarked"]]

,Name,Age,Pclass,Embarked
0,"Braund, Mr. Owen Harris",22.0,3,S
4,"Allen, Mr. William Henry",35.0,3,S
5,"Moran, Mr. James",NaN,3,Q
7,"Palsson, Master. Gosta Leonard",2.0,3,S
12,"Saundercock, Mr. William Henry",20.0,3,S
...,...,...,...,...
1290,"Conlon, Mr. Thomas Henry",31.0,3,Q
1304,"Spector, Mr. Woolf",NaN,3,S
1306,"Saether, Mr. Simon Sivertsen",38.5,3,S
1307,"Ware, Mr. Frederick",NaN,3,S


## 3. Practice on the Titanic data

The cells below combine two conditions with `&`. Remember: **every condition needs
its own brackets**, because `&` binds tighter than `==` and `>`.

In [18]:
# find out the members numbers  of 3rd class who is female
df.loc[(df["Pclass"]==3) & (df["Sex"]=="female"),["Pclass","Sex","Name"]]

,Pclass,Sex,Name
2,3,female,"Heikkinen, Miss. Laina"
8,3,female,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)"
10,3,female,"Sandstrom, Miss. Marguerite Rut"
14,3,female,"Vestrom, Miss. Hulda Amanda Adolfina"
18,3,female,"Vander Planke, Mrs. Julius (Emelia Maria Vande..."
...,...,...,...
1274,3,female,"McNamee, Mrs. Neal (Eileen O'Leary)"
1299,3,female,"Riordan, Miss. Johanna Hannah"""""
1300,3,female,"Peacock, Miss. Treasteall"
1301,3,female,"Naughton, Miss. Hannah"


In [19]:
# find out the members numbers  of 3rd class who is male

df.loc[(df["Sex"] == "male") & (df["Pclass"] == 3), :]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1290,1291,0,3,"Conlon, Mr. Thomas Henry",male,31.0,0,0,21332,7.7333,NaN,Q
1304,1305,0,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,0,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [20]:
# Female passengers older than 25
df.loc[(df["Sex"] == "female") & (df["Age"] > 25), ["Name", "Age", "Pclass"]]

,Name,Age,Pclass
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1
2,"Heikkinen, Miss. Laina",26.0,3
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,3
11,"Bonnell, Miss. Elizabeth",58.0,1
...,...,...,...
1288,"Frolicher-Stehli, Mrs. Maxmillian (Margaretha ...",48.0,1
1291,"Bonnell, Miss. Caroline",30.0,1
1302,"Minahan, Mrs. William Edward (Lillian E Thorpe)",37.0,1
1303,"Henriksson, Miss. Jenny Lovisa",28.0,3


In [21]:
# Average age of survivors
df.loc[df["Survived"]==1,"Age"].mean()


np.float64(28.93107913669065)

In [22]:
#Rich passengers (Fare > 100) who survived only find name and age
df.loc[(df["Fare"] > 100) & (df["Survived"] == 1), ["Name", "Age"]]

,Name,Age
31,"Spencer, Mrs. William Augustus (Marie Eugenie)",NaN
88,"Fortune, Miss. Mabel Helen",23.00
195,"Lurette, Miss. Elise",58.00
215,"Newell, Miss. Madeleine",31.00
258,"Ward, Miss. Anna",35.00
268,"Graham, Mrs. William Thompson (Edith Junkins)",58.00
269,"Bissette, Miss. Amelia",35.00
299,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",50.00
305,"Allison, Master. Hudson Trevor",0.92
306,"Fleming, Miss. Margaret",NaN


In [23]:
# find the name and age of Children (<15) in 3rd class
# NOTE: the class condition has to be in the filter too, not only in the comment
df.loc[(df["Age"] < 15) & (df["Pclass"] == 3), ["Name", "Age", "Pclass"]]

,Name,Age,Pclass
7,"Palsson, Master. Gosta Leonard",2.0,3
10,"Sandstrom, Miss. Marguerite Rut",4.0,3
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0,3
16,"Rice, Master. Eugene",2.0,3
24,"Palsson, Miss. Torborg Danira",8.0,3
...,...,...,...
1251,"Sage, Master. William Henry",14.5,3
1270,"Asplund, Master. Carl Edgar",5.0,3
1280,"Palsson, Master. Paul Folke",6.0,3
1283,"Abbott, Master. Eugene Joseph",13.0,3


In [24]:
# Count passengers older than 60
# .shape[0] is the number of rows left after filtering
print("Passengers older than 60:", df.loc[df["Age"] > 60, :].shape[0])

Passengers older than 60: 33


---

## 4. Task — repeat the practice on a second dataset

> *Original note: "take another dataset and make a 20 to 30 question and do a practice like this"*

Below the same ideas are practised on `nepali_data.csv` (1000 randomly generated
people with `Name`, `Salary`, `Field`, `Age`, `City`), so the conditions are tested
on data that has nothing to do with the Titanic.

In [25]:
import pandas as pd

emp = pd.read_csv("../data/nepali_data.csv")
print(emp.shape)
emp.head()

(1000, 5)


,Name,Salary,Field,Age,City
0,Sita,57072,Education,42,Kathmandu
1,Ram,115164,Finance,29,Bhaktapur
2,Prakash,59132,IT,37,Janakpur
3,Bikash,129769,Education,32,Biratnagar
4,Bikash,96931,Education,27,Kathmandu


In [26]:
# Q1. What columns and dtypes does this dataset have?
emp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Name    1000 non-null   object
 1   Salary  1000 non-null   int64 
 2   Field   1000 non-null   object
 3   Age     1000 non-null   int64 
 4   City    1000 non-null   object
dtypes: int64(2), object(3)
memory usage: 39.2+ KB


In [27]:
# Q2. People who earn more than 100000
emp.loc[emp["Salary"] > 100000]

,Name,Salary,Field,Age,City
1,Ram,115164,Finance,29,Bhaktapur
3,Bikash,129769,Education,32,Biratnagar
6,Sita,106499,Finance,46,Kathmandu
8,Hari,133411,IT,21,Chitwan
16,Sabin,119991,Medicine,58,Lalitpur
...,...,...,...,...,...
995,Nabin,107833,Finance,54,Kathmandu
996,Sunil,112121,Business,35,Nepalgunj
997,Sabin,128653,Medicine,29,Pokhara
998,Prakash,144139,Medicine,20,Butwal


In [28]:
# Q3. How many people earn more than 100000?
print("Count:", emp.loc[emp["Salary"] > 100000].shape[0])

Count: 396


In [29]:
# Q4. Name and Salary of everyone in the IT field
emp.loc[emp["Field"] == "IT", ["Name", "Salary"]]

,Name,Salary
2,Prakash,59132
8,Hari,133411
19,Roshan,79682
21,Roshan,62176
26,Prakash,51300
...,...,...
958,Anil,70667
972,Bikash,106980
979,Sabin,111551
987,Gita,60577


In [30]:
# Q5. Everyone from Kathmandu
emp.loc[emp["City"] == "Kathmandu"]

,Name,Salary,Field,Age,City
0,Sita,57072,Education,42,Kathmandu
4,Bikash,96931,Education,27,Kathmandu
6,Sita,106499,Finance,46,Kathmandu
15,Bikash,92340,Medicine,48,Kathmandu
19,Roshan,79682,IT,22,Kathmandu
...,...,...,...,...,...
945,Nabin,64350,Education,22,Kathmandu
956,Roshan,93133,IT,46,Kathmandu
969,Roshan,63694,Business,31,Kathmandu
977,Kiran,146837,Education,39,Kathmandu


In [31]:
# Q6. IT people from Kathmandu (two conditions with &)
emp.loc[(emp["Field"] == "IT") & (emp["City"] == "Kathmandu"), ["Name", "Salary", "Age"]]

,Name,Salary,Age
19,Roshan,79682,22
81,Bikash,110794,25
96,Suman,136318,55
124,Suman,138413,49
145,Sabin,99887,41
198,Bikash,58696,38
219,Nabin,138349,56
227,Suman,48705,44
283,Kiran,134084,58
414,Shyam,32015,55


In [32]:
# Q7. People younger than 30 OR older than 55 (| means OR)
emp.loc[(emp["Age"] < 30) | (emp["Age"] > 55), ["Name", "Age"]]

,Name,Age
1,Ram,29
4,Bikash,27
5,Gita,22
7,Roshan,59
8,Hari,21
...,...,...
986,Roshan,56
988,Bibek,59
992,Gita,57
997,Sabin,29


In [33]:
# Q8. People who are NOT from Pokhara (~ flips a mask)
emp.loc[~(emp["City"] == "Pokhara"), ["Name", "City"]]

,Name,City
0,Sita,Kathmandu
1,Ram,Bhaktapur
2,Prakash,Janakpur
3,Bikash,Biratnagar
4,Bikash,Kathmandu
...,...,...
994,Prakash,Dharan
995,Nabin,Kathmandu
996,Sunil,Nepalgunj
998,Prakash,Butwal


In [34]:
# Q9. People working in IT, Finance or Medicine (isin is cleaner than three | conditions)
emp.loc[emp["Field"].isin(["IT", "Finance", "Medicine"]), ["Name", "Field"]]

,Name,Field
1,Ram,Finance
2,Prakash,IT
6,Sita,Finance
8,Hari,IT
11,Shyam,Medicine
...,...,...
988,Bibek,IT
993,Hari,Finance
995,Nabin,Finance
997,Sabin,Medicine


In [35]:
# Q10. Average salary of everyone in the Engineer field
emp.loc[emp["Field"] == "Engineer", "Salary"].mean()

np.float64(85493.84090909091)

In [36]:
# Q11. Highest salary in the dataset, and who earns it
print("Highest salary:", emp["Salary"].max())
emp.loc[emp["Salary"] == emp["Salary"].max(), ["Name", "Field", "City", "Salary"]]

Highest salary: 149890


,Name,Field,City,Salary
932,Roshan,Business,Pokhara,149890


In [37]:
# Q12. Lowest salary in the Business field
emp.loc[emp["Field"] == "Business", "Salary"].min()

np.int64(21061)

In [38]:
# Q13. People earning between 50000 and 80000 (inclusive) — two conditions
emp.loc[(emp["Salary"] >= 50000) & (emp["Salary"] <= 80000), ["Name", "Salary"]]

,Name,Salary
0,Sita,57072
2,Prakash,59132
10,Nabin,78529
13,Hari,65236
14,Suresh,63573
...,...,...
976,Gita,60662
981,Nabin,71374
987,Gita,60577
989,Suresh,69372


In [39]:
# Q14. Same thing with between(), which is shorter and reads better
emp.loc[emp["Salary"].between(50000, 80000), ["Name", "Salary"]]

,Name,Salary
0,Sita,57072
2,Prakash,59132
10,Nabin,78529
13,Hari,65236
14,Suresh,63573
...,...,...
976,Gita,60662
981,Nabin,71374
987,Gita,60577
989,Suresh,69372


In [40]:
# Q15. Average age of people earning more than 120000
emp.loc[emp["Salary"] > 120000, "Age"].mean()

np.float64(40.57142857142857)

In [41]:
# Q16. How many people work in each field? (value_counts on a filtered view)
emp.loc[emp["Age"] > 40, "Field"].value_counts()

Field
Education    85
Medicine     85
Business     84
Finance      83
Engineer     80
IT           73
Name: count, dtype: int64

In [42]:
# Q17. Names that start with the letter "S"
emp.loc[emp["Name"].str.startswith("S"), ["Name", "City"]].head(10)

,Name,City
0,Sita,Kathmandu
6,Sita,Kathmandu
11,Shyam,Janakpur
14,Suresh,Butwal
16,Sabin,Lalitpur
20,Suresh,Bhaktapur
24,Suman,Butwal
30,Sunil,Dharan
33,Suresh,Nepalgunj
34,Sunil,Kathmandu


In [43]:
# Q18. Everyone whose city name contains "pur"
emp.loc[emp["City"].str.contains("pur", case=False, na=False), ["Name", "City"]].head(10)

,Name,City
1,Ram,Bhaktapur
2,Prakash,Janakpur
10,Nabin,Bhaktapur
11,Shyam,Janakpur
12,Ram,Bhaktapur
16,Sabin,Lalitpur
20,Suresh,Bhaktapur
21,Roshan,Janakpur
28,Ramesh,Janakpur
29,Nabin,Lalitpur


In [44]:
# Q19. The 5 highest paid people under 35 (filter -> sort -> head)
emp.loc[emp["Age"] < 35].sort_values("Salary", ascending=False).head(5)

,Name,Salary,Field,Age,City
932,Roshan,149890,Business,23,Pokhara
175,Prakash,149453,Finance,24,Butwal
452,Manish,149437,IT,22,Pokhara
856,Suman,149306,Finance,24,Dharan
450,Sita,148978,Engineer,30,Janakpur


In [45]:
# Q20. Update with a condition: mark everyone above 100000 as "High" earner.
# df.loc[condition, "new_column"] = value is the SAFE way to write.
emp.loc[emp["Salary"] > 100000, "SalaryBand"] = "High"
emp.loc[emp["Salary"] <= 100000, "SalaryBand"] = "Normal"
emp["SalaryBand"].value_counts()

SalaryBand
Normal    604
High      396
Name: count, dtype: int64

In [46]:
# Q21. Average salary of each band — a quick check that Q20 worked
emp.groupby("SalaryBand")["Salary"].mean()

SalaryBand
High      124125.136364
Normal     60816.925497
Name: Salary, dtype: float64

In [47]:
# Q22. Engineers older than 45 living in Kathmandu — three conditions
emp.loc[
    (emp["Field"] == "Engineer")
    & (emp["Age"] > 45)
    & (emp["City"] == "Kathmandu"),
    ["Name", "Age", "Salary"]
]

,Name,Age,Salary
72,Dipesh,46,147134
85,Shyam,54,131642
186,Sabin,50,46111
334,Aashish,55,139964
618,Manish,54,83201


### Key takeaways

| Goal | Code |
|------|------|
| Filter rows | `df.loc[condition]` |
| Filter rows + pick columns | `df.loc[condition, ["A", "B"]]` |
| AND / OR / NOT | `&` / `\|` / `~` (each condition in its own brackets) |
| One of several values | `col.isin([...])` |
| Range | `col.between(low, high)` |
| Count matches | `df.loc[cond].shape[0]` |
| Safe update | `df.loc[cond, "col"] = value` |